# Mirror 7 V5 Enhanced — Google Colab Pro (A100 High-RAM) Training Pipeline
### (Single Unified Extreme-Quality Dataset: Quantum Computing, Mathematics & Critical Thinking)

This notebook trains **Mirror 7 NativeConversationV5 Enhanced** on a Google Colab Pro **NVIDIA A100 GPU (40GB / 80GB VRAM + 83GB System RAM)** (and also supports L4, V100, T4, and CPU).

### Key Capabilities Included
- **Architecture**: Input-Conditioned Recurrent Decoder (`BiGRU` Encoder + `8-Head Dual-Scale Cross-Attention` + `Context Gating` + `Boundary Anchor Refresh` + `Sparse Top-2 MoE` + `Key-Value Memory Bank` + `RFC-3629 UTF-8 Grammar Decoding`)
- **A100 High-RAM Parameter Scaling Presets (`--model-scale`)**:
  - **`a100-pro` (Recommended Sweet Spot — `~36.7M` params)**: `embed=256, hidden=768, state=384, attn=384, heads=8, 8-Expert Top-2 MoE, 8,192 Memory Slots`.
  - **`a100-max` (Maximum Capacity — `~103.5M` params)**: `embed=384, hidden=1024, state=512, attn=512, heads=8, 16-Expert Top-2 MoE, 16,384 Memory Slots`.
  - **`medium` (`~13.8M` params)** and **`base` (`924,739` params)** also supported.
- **Single Unified Extreme-Quality Dataset**: `training/data/mirror_extreme_quality_master_v1.jsonl` (7-stage extreme-filtered across `BoltzmannEntropy/QuantumLLMInstruct`, `migtissera/Quanta`, `Qiskit/Qiskit-QuantumKatas`, `openai/gsm8k-socratic`, `mirror7_gold_anchor`, and `mirror_oasst2_curated_v1`).

## 1. Verify Colab Pro A100 GPU & High-RAM Runtime

In [ ]:
!nvidia-smi
import os, psutil, torch
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"System RAM: {ram_gb:.1f} GB | CPU Cores: {os.cpu_count()} | PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB | Compute: {props.major}.{props.minor} | BF16: {torch.cuda.is_bf16_supported()}")

## 2. Mount Google Drive & Extract `Mirror-7-main-V5-Enhanced.zip` onto Local NVMe (`/content`)
Extracting onto `/content` (Colab's fast local NVMe SSD) while saving checkpoints to `/content/drive/MyDrive` gives maximum I/O speed AND permanent Google Drive safety.

In [ ]:
import os, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')

for zip_candidate in ['/content/drive/MyDrive/Mirror-7-main-V5-Enhanced.zip', '/content/Mirror-7-main-V5-Enhanced.zip']:
    if os.path.exists(zip_candidate) and not os.path.exists('/content/Mirror-7-main'):
        print(f'Extracting {zip_candidate} to /content ...')
        with zipfile.ZipFile(zip_candidate, 'r') as zf:
            zf.extractall('/content')
        break

project_root = '/content/Mirror-7-main'
if os.path.exists(project_root):
    os.chdir(project_root)
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
print('Working Directory:', os.getcwd())

## 3. Verify Single Unified Extreme-Quality Master Dataset
The pre-built `training/data/mirror_extreme_quality_master_v1.jsonl` is already included in the ZIP. (If you ever want to re-run live HuggingFace CDN extraction, set `REBUILD_DATASET = True`).

In [ ]:
import json, os
from training.dataset import load_jsonl, summarize, validate_split_isolation

MASTER_DATASET = 'training/data/mirror_extreme_quality_master_v1.jsonl'
REBUILD_DATASET = False
if REBUILD_DATASET or not os.path.exists(MASTER_DATASET):
    from training.curate_quality_dataset import curate_dataset
    curate_dataset('training/data/mirror_oasst2_curated_v1.jsonl', MASTER_DATASET, min_score=0.78)

examples = load_jsonl(MASTER_DATASET)
validate_split_isolation(examples)
print('=== UNIFIED EXTREME-QUALITY MASTER DATASET VERIFIED ===')
print(json.dumps(summarize(examples), indent=2))

## 4. Train on A100 High-RAM (Single Unified Dataset + TF32/BF16 Tensor Cores)
- Automatically enables **A100 TF32 Tensor Cores**, **BF16 Mixed Precision**, **Fused CUDA AdamW**, **Pinned-Memory Async Transfers**, and **Zero-Data-Loss Continuation Chunking**.
- Change `MODEL_SCALE` between `'a100-pro'` (`~36.7M` params, recommended), `'a100-max'` (`~103.5M` params), `'medium'` (`~13.8M` params), or `'base'` (`924K` params).

In [ ]:
from training.gpu_train import train_gpu

MODEL_SCALE = 'a100-pro'  # Options: 'a100-pro' (~36.7M), 'a100-max' (~103.5M), 'medium' (~13.8M), 'base' (0.92M)
OUTPUT_CKPT = f'/content/drive/MyDrive/mirror7_v5_{MODEL_SCALE}_extreme_master.pt'

# Optional 2-step smoke test before full training:
_ = train_gpu(
    dataset_path=MASTER_DATASET,
    output_path='/content/smoke_test.pt',
    model_scale=MODEL_SCALE,
    smoke_test_steps=2,
    batch_size=16,
    max_bytes=512,
)
print('Smoke test passed! Starting full A100 training run...')

train_res = train_gpu(
    dataset_path=MASTER_DATASET,
    output_path=OUTPUT_CKPT,
    model_scale=MODEL_SCALE,
    epochs=8,
    batch_size=96 if MODEL_SCALE != 'a100-max' else 48,
    max_tokens_per_batch=98304,
    lr=8e-4 if MODEL_SCALE in ('a100-pro', 'a100-max') else 1e-3,
    max_bytes=1024,
    use_amp=True,
    scheduled_sampling_p=0.10,
    save_every_steps=100,
)
print('Best Checkpoint Saved To:', train_res['best_checkpoint'])

## 5. Generate First Responses (Deliberative Think-Then-Answer & Best-of-N Verification)

In [ ]:
from training.native_conversation_v5 import load_checkpoint

model, payload = load_checkpoint(train_res['best_checkpoint'], device='cuda' if torch.cuda.is_available() else 'cpu')
print(f"Loaded {payload['parameter_count']:,} parameters | Best Val Loss: {payload['metadata'].get('best_val_loss'):.4f}")

prompts = [
    ("What is a qubit and how does quantum superposition work mathematically?", "quantum_computing"),
    ("How do you construct a maximally entangled Bell state |Phi+> using quantum gates?", "quantum_computing"),
    ("Find the eigenvalues and eigenvectors of the matrix A = [[4, 1], [2, 3]] step by step.", "mathematics"),
    ("You have 9 identical-looking coins and 1 is heavier. How do you find it in 2 weighings?", "critical_thinking"),
]

for question, domain in prompts:
    res = model.generate_with_thinking(
        question,
        goal=f"{domain}_analysis",
        state={"domain": domain, "mode": "rigorous_step_by_step"},
        num_candidates=4,
        max_answer_tokens=384,
    )
    print('=' * 80)
    print(f'PROMPT [{domain}]: {question}')
    print(f'THOUGHT PLAN: {res["thought_plan"]}')
    print(f'VERIFIED RESPONSE (Score={max(res["scores"]):.2f}):\n{res["answer"]}')
